In [1]:
# just suppress all warnings when running localy you can comment out first two lines
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# =========================
# Standard Library
# =========================
import os
import re
import json
import time
import random
import hashlib
import sqlite3
from collections import defaultdict
from dataclasses import dataclass
from json import JSONDecodeError
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# =========================
# Third-Party Libraries
# =========================
import faiss
import httpx
import numpy as np
import pandas as pd
import scipdf
from jsonschema import validate, ValidationError
from openai import OpenAI, APIConnectionError, APITimeoutError, RateLimitError
from tqdm import tqdm

# =========================
# NLP / Text Processing
# =========================
import textacy.preprocessing as tprep

In [2]:
# ----------------------------
# Config
# ----------------------------
OPENAI_API_KEY = os.environ.get("OPEN_AI_API_KEY") # read api key from local environment variable
if not OPENAI_API_KEY:
    raise RuntimeError("Missing OPEN_AI_API_KEY environment variable")

client = OpenAI(api_key=OPENAI_API_KEY)# , http_client=http_client)

EMBEDDING_MODEL = "text-embedding-3-large"  # or text-embedding-3-small
LLM_MODEL = "gpt-5"

# Retrieval tuning
TOP_K = 20
MIN_CHUNK_CHARS = 200

In [3]:
# ----------------------------
# Utilities
# ----------------------------
def normalize(text: str) -> str:
    text = tprep.normalize.hyphenated_words(text or "")
    text = tprep.normalize.quotation_marks(text)
    text = tprep.normalize.unicode(text)
    text = tprep.remove.accents(text)
    # keep whitespace sane
    text = re.sub(r"\s+", " ", text).strip()
    return text

def sha256(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

SMART_QUOTES = {
    "\u201c": "\"", "\u201d": "\"",  # “ ”
    "\u2018": "'",  "\u2019": "'",   # ‘ ’
}

def extract_json_object(text: str) -> str:
    """
    Extract the first top-level JSON object {...} from a messy string
    using brace matching (safe for nested JSON).
    """
    text = text.strip()
    # Find first '{'
    start = text.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")
    # Brace matching
    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == "\"":
                in_str = False
        else:
            if ch == "\"":
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i+1]
    raise ValueError("Unbalanced braces in model output.")

def cleanup_json_text(s: str) -> str:
    # Replace smart quotes
    for k, v in SMART_QUOTES.items():
        s = s.replace(k, v)

    # Remove trailing commas before } or ]
    s = re.sub(r",\s*([}\]])", r"\1", s)

    # If model accidentally used single quotes for keys/strings (rare if you enforce JSON),
    # don't auto-convert here (can be unsafe). Prefer LLM repair in that case.
    return s

def safe_json_loads(s: str) -> dict:
    """
    Robust parse: extract JSON object, cleanup, load.
    Raises JSONDecodeError if still invalid.
    """
    obj_text = extract_json_object(s)
    obj_text = cleanup_json_text(obj_text)
    return json.loads(obj_text)

In [4]:
# ----------------------------
# Disk cache (sqlite)
# ----------------------------
class SQLiteCache:
    def __init__(self, path: str = "rag_cache.sqlite"):
        self.path = path
        self._init()

    def _init(self):
        conn = sqlite3.connect(self.path)
        cur = conn.cursor()
        cur.execute("""
        CREATE TABLE IF NOT EXISTS cache (
            key TEXT PRIMARY KEY,
            value TEXT NOT NULL,
            created_at REAL NOT NULL
        )
        """)
        conn.commit()
        conn.close()

    def get(self, key: str) -> Optional[str]:
        conn = sqlite3.connect(self.path)
        cur = conn.cursor()
        cur.execute("SELECT value FROM cache WHERE key = ?", (key,))
        row = cur.fetchone()
        conn.close()
        return row[0] if row else None

    def set(self, key: str, value: str):
        conn = sqlite3.connect(self.path)
        cur = conn.cursor()
        cur.execute(
            "INSERT OR REPLACE INTO cache(key, value, created_at) VALUES(?, ?, ?)",
            (key, value, time.time())
        )
        conn.commit()
        conn.close()

cache = SQLiteCache()

In [5]:
# ----------------------------
# Schema
# ----------------------------
EXTRACTION_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "paper_id",
        "title",
        "study_type",
        "is_nanoplastic_only",
        "has_blood_serum_plasma_biomarkers",
        "experimental_model",
        "species",
        "sex",
        "exposure_route",
        "exposure_duration",
        "dose_groups",
        "polymer_types",
        "particle_shape",
        "particle_size",
        "blood_matrix",
        "biomarkers",
        "additional_findings",
        "notes",
        "evidence"
    ],
    "properties": {
        "paper_id": {"type": "string"},
        "title": {"type": "string"},
        "study_type": {"type": "string", "enum": ["in_vivo", "in_vitro", "ex_vivo", "in_silico", "unknown"]},
        "is_nanoplastic_only": {"type": "boolean"},
        "has_blood_serum_plasma_biomarkers": {"type": "boolean"},
        "experimental_model": {"type": "string"},  # e.g., "Wistar rat", "Zebrafish", "HepG2 cells"
        "species": {"type": "string"},            # controlled later
        "sex": {"type": ["string", "null"]},      # "male"|"female"|null
        "exposure_route": {"type": ["string", "null"]},  # oral, inhalation, etc.
        "exposure_duration": {"type": "array", "items": {"type": "string"}},  # keep raw, normalize later
        "dose_groups": {"type": "array", "items": {"type": "string"}},
        "polymer_types": {"type": "array", "items": {"type": "string"}},  # "PS (polystyrene)"
        "particle_shape": {"type": ["string", "null"]},  # fiber/sphere/fragment/etc
        "particle_size": {"type": ["string", "null"]},   # keep raw (e.g. "1–5 µm")
        "blood_matrix": {"type": ["string", "null"]},    # blood/serum/plasma
        "biomarkers": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "ALT": {"type": ["string", "null"]},
                "AST": {"type": ["string", "null"]},
                "ALP": {"type": ["string", "null"]},
                "GGT": {"type": ["string", "null"]},
                "HDL": {"type": ["string", "null"]},
                "LDL": {"type": ["string", "null"]},
                "cholesterol_total": {"type": ["string", "null"]},
                "triglycerides": {"type": ["string", "null"]},
                "glucose": {"type": ["string", "null"]},
                "urea": {"type": ["string", "null"]},
                "creatinine": {"type": ["string", "null"]},
                "bilirubin": {"type": ["string", "null"]},
                "MDA": {"type": ["string", "null"]},
                "AOPP": {"type": ["string", "null"]},
                "PAB": {"type": ["string", "null"]},
                "HNE": {"type": ["string", "null"]}
            },
            "required": [
                "ALT","AST","ALP","GGT","HDL","LDL",
                "cholesterol_total","triglycerides","glucose",
                "urea","creatinine","bilirubin",
                "MDA","AOPP","PAB","HNE"
            ]
        },
        "additional_findings": {"type": "array", "items": {"type": "string"}},
        "notes": {"type": "array", "items": {"type": "string"}},
        # Evidence is the big upgrade
        "evidence": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["field", "chunk_id", "section_heading", "quote"],
                "properties": {
                    "field": {"type": "string"},  # e.g., "biomarkers.ALT" or "particle_size"
                    "chunk_id": {"type": "string"},
                    "section_heading": {"type": "string"},
                    "quote": {"type": "string"}   # short snippet
                }
            }
        }
    }
}

In [6]:
# ----------------------------
# Parsing PDFs into chunks
# ----------------------------
@dataclass
class Chunk:
    paper_id: str
    title: str
    chunk_id: str
    section_heading: str
    text: str

def parse_pdf_to_chunks(pdf_path: str) -> List[Chunk]:
    """
    Uses scipdf(GROBID). Produces paragraph-ish chunks from section text split.
    """
    article = scipdf.parse_pdf_to_dict(pdf_path)
    title = article.get("title") or Path(pdf_path).name
    paper_id = sha256(str(pdf_path))[:16]

    chunks: List[Chunk] = []
    sections = article.get("sections") or []
    for si, sec in enumerate(sections):
        heading = (sec.get("heading") or "").strip()
        text = normalize(sec.get("text") or "")
        if not text:
            continue

        # Split into pseudo-paragraphs: this is simple and works surprisingly well.
        # You can replace with a smarter splitter later.
        parts = re.split(r"(?:\n{2,}|\.\s{2,})", text)
        for pi, part in enumerate(parts):
            part = normalize(part)
            if len(part) < MIN_CHUNK_CHARS:
                continue
            cid = f"{paper_id}_s{si}_p{pi}"
            chunks.append(Chunk(
                paper_id=paper_id,
                title=title,
                chunk_id=cid,
                section_heading=heading if heading else "unknown",
                text=part
            ))
    return chunks

In [7]:
# ----------------------------
# Embeddings + FAISS
# ----------------------------
def embed_texts(texts: List[str]) -> np.ndarray:
    """
    Returns float32 matrix shape (n, d).
    Uses caching by content hash.
    """
    # Batch embedding with caching per text (simple)
    vectors = []
    to_embed = []
    to_embed_idx = []

    for i, t in enumerate(texts):
        key = f"emb:{EMBEDDING_MODEL}:{sha256(t)}"
        cached = cache.get(key)
        if cached is not None:
            vectors.append(np.frombuffer(bytes.fromhex(cached), dtype=np.float32))
        else:
            vectors.append(None)
            to_embed.append(t)
            to_embed_idx.append(i)

    if to_embed:
        resp = client.embeddings.create(model=EMBEDDING_MODEL, input=to_embed)
        # OpenAI returns embeddings in input order
        for j, item in enumerate(resp.data):
            vec = np.array(item.embedding, dtype=np.float32)
            i = to_embed_idx[j]
            vectors[i] = vec
            cache.set(
                f"emb:{EMBEDDING_MODEL}:{sha256(texts[i])}",
                vec.tobytes().hex()
            )

    mat = np.vstack(vectors).astype(np.float32)
    # Normalize for cosine similarity with inner product index
    faiss.normalize_L2(mat)
    return mat

def embed_texts(texts: List[str], batch_size: int = 64, max_retries: int = 8) -> np.ndarray:
    """
    Robust batched embedding with caching + retries.
    """
    vectors: List[Optional[np.ndarray]] = [None] * len(texts)

    # figure out which need embedding
    pending = []
    for i, t in enumerate(texts):
        key = f"emb:{EMBEDDING_MODEL}:{sha256(t)}"
        cached = cache.get(key)
        if cached is not None:
            vectors[i] = np.frombuffer(bytes.fromhex(cached), dtype=np.float32)
        else:
            pending.append(i)

    def embed_batch(batch_texts: List[str]) -> List[np.ndarray]:
        # retry wrapper
        for attempt in range(max_retries):
            try:
                resp = client.embeddings.create(
                    model=EMBEDDING_MODEL,
                    input=batch_texts,
                    timeout=60  # increase vs default
                )
                out = []
                for item in resp.data:
                    out.append(np.array(item.embedding, dtype=np.float32))
                return out
            except (APIConnectionError, APITimeoutError, httpx.ReadError, httpx.RemoteProtocolError, httpx.ConnectError) as e:
                # exponential backoff with jitter
                sleep_s = min(2 ** attempt, 60) + random.random()
                print(f"[embed retry {attempt+1}/{max_retries}] {type(e).__name__}: {e} -> sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)
                continue
            except RateLimitError as e:
                sleep_s = min(2 ** attempt, 60) + random.random()
                print(f"[rate limit] sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)
                continue
        raise RuntimeError("Embedding failed after retries.")

    # embed in batches
    for start in range(0, len(pending), batch_size):
        idx_batch = pending[start:start + batch_size]
        txt_batch = [texts[i] for i in idx_batch]
        vec_batch = embed_batch(txt_batch)

        for i, vec in zip(idx_batch, vec_batch):
            vectors[i] = vec
            cache.set(f"emb:{EMBEDDING_MODEL}:{sha256(texts[i])}", vec.tobytes().hex())

    mat = np.vstack(vectors).astype(np.float32)
    faiss.normalize_L2(mat)
    return mat

class CorpusIndex:
    def __init__(self):
        self.index = None
        self.chunks: List[Chunk] = []
        self.vecs = None

    def build(self, chunks: List[Chunk]):
        self.chunks = chunks
        texts = [c.text for c in chunks]
        self.vecs = embed_texts(texts)
        d = self.vecs.shape[1]
        self.index = faiss.IndexFlatIP(d)  # cosine via normalized vectors
        self.index.add(self.vecs)

    def search(self, query: str, top_k: int = TOP_K) -> List[Tuple[Chunk, float]]:
        qvec = embed_texts([query])
        D, I = self.index.search(qvec, top_k)
        results = []
        for score, idx in zip(D[0], I[0]):
            if idx < 0:
                continue
            results.append((self.chunks[idx], float(score)))
        return results

    def search_within_paper(self, paper_id: str, query: str, top_k: int = 20) -> List[Tuple[Chunk, float]]:
        idxs = self.paper_to_indices.get(paper_id, [])
        if not idxs:
            return []
    
        qvec = embed_texts([query])  # shape (1, d), already normalized
        q = qvec[0]                  # shape (d,)
    
        paper_vecs = self.vecs[idxs]             # shape (n, d)
        scores = paper_vecs @ q                  # cosine similarity (inner product)
    
        # take top_k indices
        k = min(top_k, len(idxs))
        top_local = np.argpartition(scores, -k)[-k:]
        top_local = top_local[np.argsort(scores[top_local])[::-1]]
    
        results = []
        for j in top_local:
            global_idx = idxs[int(j)]
            results.append((self.chunks[global_idx], float(scores[int(j)])))
        return results

In [8]:
BIOMARKER_PAT = re.compile(
    r"\b(ALT|AST|ALP|GGT|HDL|LDL|MDA|AOPP|PAB|HNE|bilirubin|cholesterol|triglycerides|glucose|urea|creatinine)\b",
    re.IGNORECASE
)
MATRIX_PAT = re.compile(r"\b(blood|serum|plasma)\b", re.IGNORECASE)

MP_PAT = re.compile(r"\b(microplast|polystyrene|polyethylene|polypropylene|PET|PVC|fiber|fragment|sphere|micron|µm)\b",
                    re.IGNORECASE)

# Common polymer list (MP literature)
POLYMER_TERMS = [
    # Abbrev + full names
    "PET", "polyethylene terephthalate",
    "PE", "polyethylene",
    "PP", "polypropylene",
    "PS", "polystyrene",
    "PVC", "polyvinyl chloride",
    "PA", "polyamide", "nylon",
    "PMMA", "polymethyl methacrylate",
    "PC", "polycarbonate",
    "PU", "polyurethane",
    "PVA", "polyvinyl alcohol",
    "PVP", "polyvinylpyrrolidone",
    "POM", "polyoxymethylene", "acetal",
    "EVA", "ethylene-vinyl acetate",
    "ABS", "acrylonitrile butadiene styrene",
    "PLA", "polylactic acid", "polylactide",
    "PTFE", "polytetrafluoroethylene",
    "PAN", "polyacrylonitrile",
    "PBT", "polybutylene terephthalate",
    "PI", "polyimide",
    "PCL", "polycaprolactone",
    # Sometimes present in textile / tire contexts
    "rubber", "elastomer",
]

def compile_terms_regex(terms, word_boundary=True):
    # Sort by length desc so "polyethylene terephthalate" matches before "PET"
    terms = sorted(set(terms), key=len, reverse=True)
    escaped = [re.escape(t) for t in terms]
    core = "|".join(escaped)
    if word_boundary:
        return re.compile(rf"\b({core})\b", re.IGNORECASE)
    else:
        return re.compile(rf"({core})", re.IGNORECASE)

POLY_PAT = compile_terms_regex(POLYMER_TERMS, word_boundary=True)

POLY_METHOD_PAT = re.compile(
    r"\b(FTIR|ATR-FTIR|Raman|micro-?FTIR|micro-?Raman|pyrolysis|Py-?GC/?MS|GC-?MS|DSC|TGA|XRD)\b",
    re.IGNORECASE
)

SIZE_PAT = re.compile(
    r"\b("
    r"d(?:10|50|90)\b|"                       # d10/d50/d90
    r"median\s+(?:particle\s+)?(?:size|diameter)\b|"
    r"mean\s+(?:particle\s+)?(?:size|diameter)\b|"
    r"average\s+(?:particle\s+)?(?:size|diameter)\b|"
    r"particle\s+size\s+distribution\b|"
    r"size\s+distribution\b|"
    r"diameter\b|"
    r"length\b|width\b|"
    r"aspect\s+ratio\b|"
    r"(?:\d+(?:\.\d+)?)\s*(?:nm|µm|um|mm)\b|" # explicit numeric + unit
    r"(?:nm|µm|um|micron(?:s)?|nanometer(?:s)?|millimeter(?:s)?)\b"
    r")",
    re.IGNORECASE
)

SHAPE_PAT = re.compile(
    r"\b("
    r"shape|morpholog(?:y|ical)|fragment(?:s)?|fiber(?:s)?|fibrous|filament(?:s)?|"
    r"film|foam|pellet(?:s)?|bead(?:s)?|microbead(?:s)?|sphere(?:s)?|spherical|"
    r"irregular|angular|elongated|"
    r"aspect\s+ratio|roundness|circularity"
    r")\b",
    re.IGNORECASE
)

MORPH_METHOD_PAT = re.compile(
    r"\b(SEM|TEM|EDS|EDX|AFM|optical\s+microscop(?:y|e)|microscop(?:y|e)|image\s+analysis)\b",
    re.IGNORECASE
)

def count_matches(pat: re.Pattern, text: str, cap: int = 3) -> int:
    return min(len(pat.findall(text)), cap)

def rerank_filter(results):
    rescored = []
    for ch, score in results:
        t = ch.text

        bonus = 0.0
        # biomarkers / matrices
        if BIOMARKER_PAT.search(t): bonus += 0.15
        if MATRIX_PAT.search(t): bonus += 0.15
        if MP_PAT.search(t): bonus += 0.10

        # particle characterization emphasis
        bonus += 0.08 * count_matches(SIZE_PAT, t)         # up to +0.24
        bonus += 0.06 * count_matches(SHAPE_PAT, t)        # up to +0.18
        bonus += 0.07 * count_matches(POLY_PAT, t)         # up to +0.21

        # method mentions help too
        if POLY_METHOD_PAT.search(t): bonus += 0.10
        if MORPH_METHOD_PAT.search(t): bonus += 0.08

        rescored.append((ch, score + bonus))

    rescored.sort(key=lambda x: x[1], reverse=True)
    return rescored

In [9]:
# ----------------------------
# LLM extraction (strict JSON + evidence)
# ----------------------------
SYSTEM_PROMPT = """You are a careful scientific information extraction engine specialized in microplastic toxicology.

Rules:
- You must ONLY use facts explicitly stated in the provided context.
- If a value is missing or not explicitly stated, use null (or empty list where appropriate).
- Return ONLY valid JSON. No markdown, no commentary, no trailing commas.
- All strings must be valid JSON strings (escape internal quotes with backslash).
- Do NOT invent results. Do NOT infer biomarker direction unless it is explicitly stated.
- Evidence is REQUIRED: for every non-null biomarker or key variable (polymer_types, particle_size, exposure_route, dose_groups, blood_matrix),
  include at least one evidence item with a short quote copied from the context, plus its chunk_id and section_heading.
- Evidence.quote must be <= 240 characters, single line (no newline characters).
- If the study is nanoplastic-only (no microplastics), set is_nanoplastic_only=true.
- If there are no blood/serum/plasma biochemical or oxidative stress biomarkers, set has_blood_serum_plasma_biomarkers=false.

Output must follow the exact schema (no extra keys).
"""

def make_extraction_prompt(paper_id: str, title: str, context_chunks: List[Chunk]) -> List[Dict[str, str]]:
    context = []
    for c in context_chunks:
        context.append(
            f"[chunk_id={c.chunk_id}] [section={c.section_heading}]\n{c.text}"
        )
    context_text = "\n\n".join(context)

    user = {
        "role": "user",
        "content": json.dumps({
            "paper_id": paper_id,
            "title": title,
            "task": "Extract microplastic exposure details and blood/serum/plasma biomarkers with evidence mapping.",
            "context": context_text,
            "schema_hint": EXTRACTION_SCHEMA  # ok to include; helps follow structure
        })
    }
    return [{"role": "system", "content": SYSTEM_PROMPT}, user]

def llm_extract(paper_id: str, title: str, context_chunks: List[Chunk]) -> Dict[str, Any]:
    cache_key = f"extract:{LLM_MODEL}:{paper_id}:{sha256(''.join([c.chunk_id for c in context_chunks]))}"
    cached = cache.get(cache_key)
    if cached:
        return json.loads(cached)

    messages = make_extraction_prompt(paper_id, title, context_chunks)
    # print(messages)
    resp = client.responses.create(
        model=LLM_MODEL,
        input=messages,
        reasoning={"effort": "medium"}
    )
    # NOTE: adjust path if your SDK response shape differs
    text = resp.output[1].content[0].text  # you may need to adapt for your SDK version
    obj = safe_json_loads(text)

    cache.set(cache_key, json.dumps(obj))
    return obj

def llm_extract(paper_id: str, title: str, context_chunks: List[Chunk]) -> Dict[str, Any]:
    cache_key = f"extract:{LLM_MODEL}:{paper_id}:{sha256(''.join([c.chunk_id for c in context_chunks]))}"
    cached = cache.get(cache_key)
    if cached:
        return json.loads(cached)

    messages = make_extraction_prompt(paper_id, title, context_chunks)
    resp = client.responses.create(
        model=LLM_MODEL,
        input=messages,
        reasoning={"effort": "medium"}
    )

    text = resp.output[1].content[0].text  # adjust if needed

    try:
        obj = safe_json_loads(text)
    except (JSONDecodeError, ValueError) as e:
        # Repair via LLM
        obj = repair_invalid_json(paper_id, title, bad_text=text, error=str(e))

    cache.set(cache_key, json.dumps(obj))
    return obj

def repair_invalid_json(paper_id: str, title: str, bad_text: str, error: str) -> Dict[str, Any]:
    """
    Single repair attempt when schema validation fails.
    """
    repair_key = f"repair:{LLM_MODEL}:{paper_id}:{sha256(bad_text+error)}"
    cached = cache.get(repair_key)
    if cached:
        return json.loads(cached)

    repair_prompt = [
        {"role": "system", "content": "You fix JSON to match the given schema. Return ONLY valid JSON. No commentary."},
        {"role": "user", "content": json.dumps({
            "paper_id": paper_id,
            "title": title,
            "schema": EXTRACTION_SCHEMA,
            "validation_error": error,
            "bad_output": bad_text
        })}
    ]
    resp = client.responses.create(
        model=LLM_MODEL,
        input=repair_prompt,
        reasoning={"effort": "low"}
    )
    text = resp.output[1].content[0].text
    obj = safe_json_loads(text)

    cache.set(repair_key, json.dumps(obj))
    return obj

def repair_invalid_json(paper_id: str, title: str, bad_text: str, error: str) -> Dict[str, Any]:
    repair_key = f"repair:{LLM_MODEL}:{paper_id}:{sha256(bad_text+error)}"
    cached = cache.get(repair_key)
    if cached:
        return json.loads(cached)

    repair_prompt = [
        {"role": "system", "content":
         "You are a JSON repair tool. "
         "Return ONLY valid JSON that matches the schema exactly. "
         "No markdown, no commentary, no extra keys. "
         "Ensure all quotes inside strings are escaped properly."},
        {"role": "user", "content": json.dumps({
            "paper_id": paper_id,
            "title": title,
            "schema": EXTRACTION_SCHEMA,
            "error": error,
            "bad_output": bad_text
        })}
    ]
    resp = client.responses.create(
        model=LLM_MODEL,
        input=repair_prompt,
        reasoning={"effort": "low"}
    )
    text = resp.output[1].content[0].text
    obj = safe_json_loads(text)  # this should now be valid JSON
    cache.set(repair_key, json.dumps(obj))
    return obj

def validate_or_repair(paper_id: str, title: str, obj: Dict[str, Any], raw_text_if_any: Optional[str]=None) -> Dict[str, Any]:
    try:
        validate(instance=obj, schema=EXTRACTION_SCHEMA)
        return obj
    except ValidationError as e:
        if raw_text_if_any is None:
            raw_text_if_any = json.dumps(obj)
        repaired = repair_invalid_json(paper_id, title, raw_text_if_any, str(e))
        validate(instance=repaired, schema=EXTRACTION_SCHEMA)
        return repaired

In [10]:
# ----------------------------
# End-to-end run
# ----------------------------
def build_corpus_index_from_pdfs(pdf_dir: str) -> Tuple[CorpusIndex, Dict[str, Dict[str, Any]]]:
    pdfs = sorted([str(p) for p in Path(pdf_dir).glob("*.pdf")])
    all_chunks: List[Chunk] = []
    papers_meta: Dict[str, Dict[str, Any]] = {}

    for pdf in tqdm(pdfs, desc="Parsing PDFs"):
        chunks = parse_pdf_to_chunks(pdf)
        if not chunks:
            continue
        paper_id = chunks[0].paper_id
        title = chunks[0].title
        papers_meta[paper_id] = {"title": title, "pdf_path": pdf}
        all_chunks.extend(chunks)

    corpus = CorpusIndex()
    corpus.build(all_chunks)
    
    # NEW: map paper_id -> indices into corpus.chunks / corpus.vecs
    paper_to_indices = defaultdict(list)
    for i, ch in enumerate(corpus.chunks):
        paper_to_indices[ch.paper_id].append(i)

    # attach to corpus for later
    corpus.paper_to_indices = dict(paper_to_indices)
    
    return corpus, papers_meta

# search one query at the time and retrieve from all chunks all papers
def retrieve_context_for_paper(
    corpus: CorpusIndex, 
    paper_id: str, 
    query: str, top_k: int = TOP_K) -> List[Chunk]:
    results = corpus.search(query, top_k=top_k)
    # Filter to this paper + rerank
    paper_hits = [(c, s) for (c, s) in results if c.paper_id == paper_id]
    paper_hits = rerank_filter(paper_hits)
    return [c for (c, _) in paper_hits[:top_k]]

# search multiple queries at the time and retrieve from all chunks all papers
def retrieve_context_for_paper_multi(
    corpus: CorpusIndex,
    paper_id: str,
    top_k_each: int = 8,         # modest per-query cap (6–10 is typical)
    min_each: int = 4,           # guarantee coverage from each query
    max_total: int = 30          # hard cap so you don't send too much
) -> List[Chunk]:

    queries = [
        # 1) Exposure design / dosing
        "exposure route administered gavage oral ingestion inhalation dermal injection "
        "dose concentration mg/kg µg/kg ng/L µg/L mg/L ppm % w/v "
        "treatment group control exposure duration hours days weeks",

        # 2) Particle characterization
        "microplastic particle characterization size distribution diameter radius length width "
        "µm um nm d10 d50 d90 median mean range "
        "shape morphology spherical sphere bead fiber fibrous fragment irregular film pellet "
        "aspect ratio",

        # 3) Polymer identity
        "polymer type plastic type polystyrene polyethylene polypropylene PET PVC PA PS PE PP "
        "FTIR ATR Raman spectroscopy pyrolysis GC-MS DSC SEM EDS",

        # 4) Blood/serum/plasma biomarkers and oxidative stress
        "blood serum plasma biochemical parameters clinical chemistry "
        "ALT AST ALP GGT bilirubin creatinine urea glucose cholesterol triglycerides HDL LDL "
        "oxidative stress antioxidant prooxidant MDA SOD CAT GPx GSH AOPP PAB HNE"
    ]

    # Store hits per query (after filtering to the paper and reranking)
    per_query_hits: List[List[Tuple[Chunk, float]]] = []

    for q in queries:
        res = corpus.search(q, top_k=TOP_K)
        paper_hits = [(c, s) for (c, s) in res if c.paper_id == paper_id]
        paper_hits = rerank_filter(paper_hits)
        per_query_hits.append(paper_hits[:top_k_each])

    # 1) First pass: take at least `min_each` from each query (deduped)
    seen = set()
    selected: List[Tuple[Chunk, float]] = []

    for hits in per_query_hits:
        for ch, sc in hits[:min_each]:
            if ch.chunk_id not in seen:
                seen.add(ch.chunk_id)
                selected.append((ch, sc))

    # 2) Second pass: fill remaining slots with best remaining chunks across all queries
    remaining: List[Tuple[Chunk, float]] = []
    for hits in per_query_hits:
        for ch, sc in hits:
            if ch.chunk_id not in seen:
                remaining.append((ch, sc))

    remaining.sort(key=lambda x: x[1], reverse=True)

    for ch, sc in remaining:
        if len(selected) >= max_total:
            break
        seen.add(ch.chunk_id)
        selected.append((ch, sc))

    # Final sort for cleanliness (optional)
    selected.sort(key=lambda x: x[1], reverse=True)

    return [c for c, _ in selected]

# search multiple queries at the time and retrieve from all chunks one paper at time
def retrieve_context_for_paper_multi(corpus: CorpusIndex, paper_id: str, top_k_each: int = 10) -> List[Chunk]:
    queries = [
        "exposure route administered gavage oral ingestion inhalation dermal injection "
        "dose concentration mg/kg µg/kg ng/L µg/L mg/L ppm % w/v "
        "treatment group control exposure duration hours days weeks",

        "microplastic particle characterization size distribution diameter radius length width "
        "µm um nm d10 d50 d90 median mean range "
        "shape morphology spherical sphere bead fiber fibrous fragment irregular film pellet "
        "aspect ratio",

        "polymer type plastic type polystyrene polyethylene polypropylene PET PVC PA PS PE PP "
        "FTIR ATR Raman spectroscopy pyrolysis GC-MS DSC SEM EDS",

        "blood serum plasma biochemical parameters clinical chemistry "
        "ALT AST ALP GGT bilirubin creatinine urea glucose cholesterol triglycerides HDL LDL "
        "oxidative stress antioxidant prooxidant MDA SOD CAT GPx GSH AOPP PAB HNE"
    ]

    seen = set()
    collected: List[Tuple[Chunk, float]] = []

    for q in queries:
        res = corpus.search_within_paper(paper_id, q, top_k=top_k_each * 3)  # pull a bit more, then rerank
        res = rerank_filter(res)
        for ch, sc in res[:top_k_each]:
            if ch.chunk_id not in seen:
                seen.add(ch.chunk_id)
                collected.append((ch, sc))

    collected.sort(key=lambda x: x[1], reverse=True)
    return [c for c, _ in collected]

In [11]:
pdf_dir = "./biomed_params_MP_Ivana_jul_2025/"

In [12]:
corpus, papers_meta = build_corpus_index_from_pdfs(pdf_dir)

Parsing PDFs: 100%|██████████| 87/87 [04:41<00:00,  3.23s/it]


In [13]:
# if particles sizes do no exist recheck again
def needs_size_fix(context_chunks: List[Chunk], obj: Dict[str, Any]) -> bool:
    if obj.get("particle_size"):
        return False
    ctx = " ".join(c.text for c in context_chunks)
    return bool(SIZE_PAT.search(ctx))

def extract_size_polymer_shape_only(paper_id: str, title: str, context_chunks: List[Chunk]) -> Dict[str, Any]:
    context = "\n\n".join(
        [f"[chunk_id={c.chunk_id}] [section={c.section_heading}]\n{c.text}" for c in context_chunks]
    )
    prompt = [
        {"role":"system","content":
         "Extract ONLY particle_size, particle_shape, polymer_types from the context. "
         "Return ONLY valid JSON with keys: particle_size, particle_shape, polymer_types, evidence. "
         "Evidence items must include field, chunk_id, section_heading, quote. "
         "Escape quotes properly. Quotes must be single-line and <= 240 chars."},
        {"role":"user","content": json.dumps({"paper_id": paper_id, "title": title, "context": context})}
    ]
    resp = client.responses.create(model=LLM_MODEL, input=prompt, reasoning={"effort":"low"})
    txt = resp.output[1].content[0].text
    return safe_json_loads(txt)

In [14]:
def merge_size_polymer_shape(main_obj: Dict[str, Any], patch_obj: Dict[str, Any]) -> Dict[str, Any]:
    # Only fill missing fields
    if not main_obj.get("particle_size") and patch_obj.get("particle_size"):
        main_obj["particle_size"] = patch_obj["particle_size"]

    if not main_obj.get("particle_shape") and patch_obj.get("particle_shape"):
        main_obj["particle_shape"] = patch_obj["particle_shape"]

    # polymer_types: merge unique
    if patch_obj.get("polymer_types"):
        existing = main_obj.get("polymer_types") or []
        merged = list(dict.fromkeys(existing + patch_obj["polymer_types"]))
        main_obj["polymer_types"] = merged

    # evidence: append, but dedupe by (field, chunk_id, quote)
    if patch_obj.get("evidence"):
        existing_ev = main_obj.get("evidence") or []
        seen = {(e.get("field"), e.get("chunk_id"), e.get("quote")) for e in existing_ev}
        for e in patch_obj["evidence"]:
            key = (e.get("field"), e.get("chunk_id"), e.get("quote"))
            if key not in seen:
                existing_ev.append(e)
                seen.add(key)
        main_obj["evidence"] = existing_ev

    return main_obj

In [15]:
rows = []
for paper_id, meta in tqdm(papers_meta.items(), desc="Extracting"):
    try:
        title = meta["title"]
        print(title)
        context_chunks = retrieve_context_for_paper_multi(corpus, paper_id)
    
        # If retrieval fails, fall back to a small set of chunks from Methods/Results if available
        if not context_chunks:
            context_chunks = [c for c in corpus.chunks if c.paper_id == paper_id][:10]
    
        # Extract + validate
        obj = llm_extract(paper_id, title, context_chunks)
        obj = validate_or_repair(paper_id, title, obj, raw_text_if_any=json.dumps(obj))
        
        # ---- Fix: targeted fallback extraction for missing particle info ----
        if needs_size_fix(context_chunks, obj) or (not obj.get("polymer_types")) or (obj.get("particle_shape") is None):
            patch = extract_size_polymer_shape_only(paper_id, title, context_chunks)
            obj = merge_size_polymer_shape(obj, patch)
        
            # Optional but recommended: schema validate again after merge
            obj = validate_or_repair(paper_id, title, obj, raw_text_if_any=json.dumps(obj))
        # --------------------------------------------------------------------
        
        rows.append(obj)
    except Exception as e:
        rows.append({
            "paper_id": paper_id,
            "title": meta["title"],
            "study_type": "unknown",
            "is_nanoplastic_only": False,
            "has_blood_serum_plasma_biomarkers": False,
            "experimental_model": "",
            "species": "",
            "sex": None,
            "exposure_route": None,
            "exposure_duration": [],
            "dose_groups": [],
            "polymer_types": [],
            "particle_shape": None,
            "particle_size": None,
            "blood_matrix": None,
            "biomarkers": {k: None for k in EXTRACTION_SCHEMA["properties"]["biomarkers"]["properties"].keys()},
            "additional_findings": [],
            "notes": [f"EXTRACTION_FAILED: {type(e).__name__}: {e}"],
            "evidence": []
        })

Extracting:   0%|          | 0/87 [00:00<?, ?it/s]

Impact of dietary exposure to polyester microfibers on hematology, serology and histology in a mouse model


Extracting:   1%|          | 1/87 [01:50<2:38:45, 110.77s/it]

Lipidomics and transcriptomics insight into impacts of microplastics exposure on hepatic lipid metabolism in mice


Extracting:   2%|▏         | 2/87 [03:27<2:25:11, 102.49s/it]

...

Extracting:  98%|█████████▊| 85/87 [2:12:24<03:08, 94.46s/it]

Long-Term Exposure to Polystyrene Microspheres and High-Fat Diet-Induced Obesity in Mice: Evaluating a Role for Microbiota Dysbiosis


Extracting:  99%|█████████▉| 86/87 [2:13:42<01:29, 89.50s/it]

Assessment of Nonalcoholic Fatty Liver Disease Symptoms and Gut-Liver Axis Status in Zebrafish after Exposure to Polystyrene Microplastics and Oxytetracycline, Alone and in Combination


Extracting: 100%|██████████| 87/87 [2:14:29<00:00, 92.76s/it]


In [16]:
df = pd.json_normalize(rows)

In [17]:
# Create a modern NumPy generator
rng = np.random.default_rng(seed=42)

# Pass the generator to the sample method
df_10 = df.sample(n=10, random_state=rng)

In [18]:
evidence_df = df[["paper_id", "title", "evidence"]].explode("evidence")

In [19]:
evidence_df[((evidence_df['paper_id']=='c4fe32b245be3c4d')&\
             (evidence_df['evidence'].map(lambda x: x.get('field')=='creatinine' \
                                          if type(x)==dict else False)))]\
['evidence'].iloc[0]

{'field': 'creatinine',
 'chunk_id': 'c4fe32b245be3c4d_s16_p0',
 'section_heading': 'Parameter',
 'quote': 'led to an increase in this parameter in the P2 and P3 groups relative to the Q group... and in P3 compared to the P1 group'}

In [20]:
df.columns

Index(['paper_id', 'title', 'study_type', 'is_nanoplastic_only',
       'has_blood_serum_plasma_biomarkers', 'experimental_model', 'species',
       'sex', 'exposure_route', 'exposure_duration', 'dose_groups',
       'polymer_types', 'particle_shape', 'particle_size', 'blood_matrix',
       'additional_findings', 'notes', 'evidence', 'biomarkers.ALT',
       'biomarkers.AST', 'biomarkers.ALP', 'biomarkers.GGT', 'biomarkers.HDL',
       'biomarkers.LDL', 'biomarkers.cholesterol_total',
       'biomarkers.triglycerides', 'biomarkers.glucose', 'biomarkers.urea',
       'biomarkers.creatinine', 'biomarkers.bilirubin', 'biomarkers.MDA',
       'biomarkers.AOPP', 'biomarkers.PAB', 'biomarkers.HNE'],
      dtype='object')

In [21]:
df[['paper_id', 'title', 'study_type', 'is_nanoplastic_only',
       'has_blood_serum_plasma_biomarkers', 'experimental_model', 'species',
       'sex', 'exposure_route', 'exposure_duration', 'dose_groups',
       'polymer_types', 'particle_shape', 'particle_size', 'blood_matrix', 
    'biomarkers.ALT',
       'biomarkers.AST', 'biomarkers.ALP', 'biomarkers.GGT', 'biomarkers.HDL',
       'biomarkers.LDL', 'biomarkers.cholesterol_total',
       'biomarkers.triglycerides', 'biomarkers.glucose', 'biomarkers.urea',
       'biomarkers.creatinine', 'biomarkers.bilirubin', 'biomarkers.MDA',
       'biomarkers.AOPP', 'biomarkers.PAB', 'biomarkers.HNE']]\
.to_csv("mp_rag_extraction_results.csv", index=False)